In [1]:
import chromadb
import pandas as pd

from embeders import OllamaEmbeddingFunction

from tqdm import tqdm


In [ ]:
client = chromadb.PersistentClient(
    path=r".\.chroma_db"
)

# Use Ollama for both add and query embeddings
ef = OllamaEmbeddingFunction(model="mxbai-embed-large", host="http://127.0.0.1:11434")
collection = client.get_or_create_collection(name="checks_v2", embedding_function=ef)


Starting script


In [16]:
df_pubs = pd.read_excel("./2023_Completo_redem_0304.xlsx")

df_pubs.head()


,Date,Profile,nome_registro,partido,bloco,Message,Number of Likes,Number of comments,"Number of Reactions, Comments & Shares",Link,cargo,Post interaction rate,Message-ID,Profile-ID
0,2023-12-31 23:29:46.000,Dandara Tonantzin,Dandara Tonantzin Silva Castro,PT,Governo,Que 2024 seja um ano de grandes feitos e reali...,1375,42,1417,https://www.instagram.com/p/C1inn27t2tw/,deputado,"0,02",17956241519710050,507629575
1,2023-12-31 23:25:44.000,Katarina Feitoza,Katarina Feitoza Lima Santana,PSD,Centro,"Que 2024 venha com amor e paz, permitindo que ...",1140,60,1200,https://www.instagram.com/p/C1inKTWOrzT/,deputado,"0,03",18036335929638580,463726350
2,2023-12-31 23:23:34.000,Fred Linhares,Davys Frederico Teixeira Linhares,REPUBLICANOS,Centro,Desejamos um Feliz Ano Novo pra vocês. Que sej...,0,126,126,https://www.instagram.com/p/C1im6abtYtl/,deputado,0,18043426255590112,3968408902
3,2023-12-31 23:22:17.000,Célio Studart,Celio Studart Barbosa,PSD,Centro,É… ganhou a cor branca! Que venha a paz! Feliz...,1962,153,2115,https://www.instagram.com/p/C1imxDfrpuL/,deputado,0,18406971835000288,327375969
4,2023-12-31 23:21:16.000,Erika Hilton,Erika Santos Silva,PSOL,Governo,Meu 2023 não caberia em apenas uma retrospecti...,46753,1278,48031,https://www.instagram.com/reel/C1ikXozA8eS/,deputado,"0,02",17974508666507210,1280403287


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

BATCH_SIZE = 64  # You can adjust this based on your resources

def process_batch(batch_df):
    messages = batch_df["Message"].tolist()
    res = collection.query(query_texts=messages, n_results=10)
    results = []
    for idx, message in enumerate(messages):
        distances = [d[0] for d in res["distances"][idx]]
        min_distance = min(distances)
        results.append({
            "message": message,
            "min_distance": min_distance,
            "min_dist_document": res["documents"][idx][0],
            "doc_id": res["ids"][idx][0],
            "pub_id": batch_df.iloc[idx]["Message-ID"]
        })
    return results

df_res_list = []
batches = [df_pubs.iloc[i:i+BATCH_SIZE] for i in range(0, len(df_pubs), BATCH_SIZE)]

with ThreadPoolExecutor() as executor:
    futures = [executor.submit(process_batch, batch) for batch in batches]
    for f in tqdm(as_completed(futures), total=len(futures), desc="Processing batches"):
        df_res_list.extend(f.result())
        df_res = pd.DataFrame(df_res_list)
        df_res.to_csv("df_res.csv", index=False)


Processing batches:   0%|          | 0/5788 [00:16<?, ?it/s]
